# 02 · Exploratory Data Analysis (EDA)

Membaca Parquet hasil `01_ingest_bts` dengan **DuckDB** (query SQL langsung ke ±21 juta baris tanpa memuat semuanya ke RAM).

Pertanyaan yang dijawab:
1. Seberapa bersih datanya? (nilai kosong, batal, divert)
2. Bagaimana distribusi delay dan **kelas kompensasi PM 89/2015**? Seberapa timpang kelas 5?
3. Kapan, di maskapai mana, dan di bandara mana delay paling sering terjadi?
4. Variabel apa yang layak jadi fitur model?

Semua grafik disimpan ke `figures/` di Drive (untuk paper & PPT), ringkasan ke `eda_stats.json`.

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Data BTS 2023-2025'
except ImportError:
    BASE = './data'

import os, json
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

GLOB = os.path.join(BASE, 'parquet', '*', '*.parquet').replace(os.sep, '/')
FIG_DIR = os.path.join(BASE, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Warna mengikuti UI_RULES.md
BLUE, BLUE2, ORANGE, GOLD, GREY = '#1E3A8A', '#3B82F6', '#F97316', '#F5B700', '#94A3B8'
CLASS_COLORS = ['#BFDBFE', '#93C5FD', '#60A5FA', '#3B82F6', '#1E3A8A', ORANGE]  # kelas 5 selalu orange
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .25, 'axes.titleweight': 'bold'})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight')
    plt.show()

con = duckdb.connect()

### Definisi kelas kompensasi PM 89/2015

Delay = **keterlambatan berangkat** (`DepDelayMinutes`, nilai negatif sudah 0 dari BTS).
Penerbangan **batal / divert dikeluarkan** (pembatalan punya aturan kompensasi sendiri).

| Kelas | Delay |
|---|---|
| 0 | < 30 menit (bukan delay menurut PM 89) |
| 1 | 30–60 |
| 2 | 61–120 |
| 3 | 121–180 |
| 4 | 181–240 |
| 5 | > 240 (ganti rugi) |

In [ ]:
# shortcut: aturan kelas diduplikasi di 03_model & web; pindahkan ke satu modul kalau mulai berubah-ubah
con.sql(f'''
CREATE OR REPLACE VIEW flights AS
SELECT *,
       (CRSDepTime // 100) % 24 AS dep_hour,
       CASE WHEN DepDelayMinutes < 30  THEN 0
            WHEN DepDelayMinutes <= 60  THEN 1
            WHEN DepDelayMinutes <= 120 THEN 2
            WHEN DepDelayMinutes <= 180 THEN 3
            WHEN DepDelayMinutes <= 240 THEN 4
            ELSE 5 END AS pm89_class
FROM read_parquet('{GLOB}')
''')
# ops = penerbangan yang benar-benar beroperasi (basis model)
con.sql('CREATE OR REPLACE VIEW ops AS SELECT * FROM flights WHERE Cancelled = 0 AND Diverted = 0 AND DepDelayMinutes IS NOT NULL')
con.sql('SELECT COUNT(*) AS semua, (SELECT COUNT(*) FROM ops) AS beroperasi FROM flights').df()

## 2. Kualitas data

In [ ]:
cols = [r[0] for r in con.sql('DESCRIBE flights').fetchall()]
nulls = con.sql('SELECT ' + ', '.join(f'ROUND(100.0 * COUNT(*) FILTER (WHERE "{c}" IS NULL) / COUNT(*), 2) AS "{c}"' for c in cols) + ' FROM flights').df().T
nulls.columns = ['% kosong']
nulls[nulls['% kosong'] > 0].sort_values('% kosong', ascending=False)

Catatan yang diharapkan:
- `DepDelay*` kosong ≈ penerbangan batal (tidak pernah berangkat) → wajar.
- `CarrierDelay` dkk. kosong untuk penerbangan yang tidak delay ≥ 15 menit → wajar, dan kolom ini **bukan fitur** (baru diketahui setelah kejadian).

In [ ]:
con.sql('''
SELECT COUNT(*) AS total,
       SUM(Cancelled)::INT AS batal, ROUND(100 * AVG(Cancelled), 2) AS pct_batal,
       SUM(Diverted)::INT AS divert, ROUND(100 * AVG(Diverted), 2) AS pct_divert,
       COUNT(DISTINCT Reporting_Airline) AS maskapai,
       COUNT(DISTINCT Origin) AS bandara_asal,
       COUNT(DISTINCT Origin || '-' || Dest) AS rute
FROM flights
''').df()

## 3. Distribusi delay

In [ ]:
con.sql('''
SELECT ROUND(AVG(DepDelayMinutes), 1) AS rata2,
       quantile_cont(DepDelayMinutes, [0.5, 0.75, 0.9, 0.95, 0.99]) AS p50_p75_p90_p95_p99,
       MAX(DepDelayMinutes) AS maks,
       ROUND(100 * AVG((DepDelayMinutes = 0)::INT), 1) AS pct_tepat_waktu_atau_lebih_awal
FROM ops
''').df()

In [ ]:
h = con.sql('SELECT LEAST(DepDelayMinutes, 600) // 10 * 10 AS b, COUNT(*) n FROM ops GROUP BY 1 ORDER BY 1').df()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(h.b, h.n, width=9, color=BLUE2)
ax.set_yscale('log')
for x in (30, 60, 120, 180, 240):
    ax.axvline(x, color=ORANGE, ls='--', lw=1)
ax.set(title='Distribusi delay berangkat (skala log; garis = batas kelas PM 89)',
       xlabel='Delay berangkat (menit, ≥600 digabung)', ylabel='Jumlah penerbangan')
save(fig, '01_distribusi_delay.png')

Distribusi **sangat miring ke kanan** (*right-skewed*): mayoritas tepat waktu, ekor panjang sampai ratusan menit.
Ini alasan skenario **S2 (target log)** di Pendekatan A (regresi).

## 4. Kelas kompensasi PM 89 — seberapa timpang?

In [ ]:
cls = con.sql('SELECT pm89_class AS kelas, COUNT(*) AS n FROM ops GROUP BY 1 ORDER BY 1').df()
cls['pct'] = (100 * cls.n / cls.n.sum()).round(3)
cls['rasio_vs_kelas_0'] = (cls.n.iloc[0] / cls.n).round(1)
display(cls) if 'display' in globals() else print(cls)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(cls.kelas.astype(str), cls.pct, color=CLASS_COLORS)
ax.bar_label(bars, labels=[f'{p:.2f}%' for p in cls.pct], padding=3)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title='Proporsi kelas kompensasi PM 89/2015', xlabel='Kelas', ylabel='% penerbangan beroperasi')
save(fig, '02_distribusi_kelas.png')

Kelas 5 sangat jarang dibanding kelas 0 → **imbalanced**. Akurasi saja akan menyesatkan;
karena itu metrik utama = **macro F1** dan **recall kelas 5**, dan skenario **S4 (class weight balanced)** diuji.

## 5. Tren bulanan

In [ ]:
m = con.sql('''
SELECT make_date(Year, Month, 1) AS bulan, COUNT(*) AS flights,
       100 * AVG((pm89_class >= 1)::INT) AS pct_delay30,
       100 * AVG((pm89_class = 5)::INT) AS pct_kelas5
FROM ops GROUP BY 1 ORDER BY 1
''').df()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(m.bulan, m.pct_delay30, color=BLUE, marker='o', ms=3, label='% delay ≥ 30 menit (kelas ≥1)')
ax2 = ax.twinx()
ax2.plot(m.bulan, m.pct_kelas5, color=ORANGE, marker='o', ms=3, label='% kelas 5 (> 240 menit)')
ax.set(title='Tren bulanan keterlambatan', ylabel='% delay ≥ 30 menit')
ax2.set_ylabel('% kelas 5', color=ORANGE); ax2.spines['right'].set_visible(True); ax2.grid(False)
fig.legend(loc='upper left', bbox_to_anchor=(0.06, 0.92))
save(fig, '03_tren_bulanan.png')

## 6. Maskapai

In [ ]:
a = con.sql('''
SELECT Reporting_Airline AS maskapai, COUNT(*) AS flights,
       ROUND(100 * AVG((pm89_class >= 1)::INT), 2) AS pct_delay30,
       ROUND(100 * AVG((pm89_class = 5)::INT), 3) AS pct_kelas5,
       ROUND(AVG(DepDelayMinutes), 1) AS rata2_delay
FROM ops GROUP BY 1 ORDER BY pct_delay30 DESC
''').df()
display(a) if 'display' in globals() else print(a)

fig, ax = plt.subplots(figsize=(9, 5))
s = a.sort_values('pct_delay30')
ax.barh(s.maskapai, s.pct_delay30, color=BLUE2)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title='% penerbangan delay ≥ 30 menit per maskapai', xlabel='% delay ≥ 30 menit')
save(fig, '04_maskapai.png')

## 7. Bandara asal (min. 20.000 penerbangan)

In [ ]:
o = con.sql('''
SELECT Origin AS bandara, ANY_VALUE(OriginState) AS state, COUNT(*) AS flights,
       ROUND(100 * AVG((pm89_class >= 1)::INT), 2) AS pct_delay30,
       ROUND(100 * AVG((pm89_class = 5)::INT), 3) AS pct_kelas5
FROM ops GROUP BY 1 HAVING COUNT(*) >= 20000 ORDER BY pct_delay30 DESC LIMIT 15
''').df()
o

## 8. Hari × jam jadwal berangkat

In [ ]:
import numpy as np
hd = con.sql('''
SELECT DayOfWeek AS dow, dep_hour AS jam, 100 * AVG((pm89_class >= 1)::INT) AS pct
FROM ops GROUP BY 1, 2
''').df().pivot(index='dow', columns='jam', values='pct').reindex(columns=range(24))

fig, ax = plt.subplots(figsize=(12, 3.8))
im = ax.imshow(hd.values, aspect='auto', cmap='Blues')
ax.set_yticks(range(7), ['Sen', 'Sel', 'Rab', 'Kam', 'Jum', 'Sab', 'Min'])
ax.set_xticks(range(24)); ax.grid(False)
ax.set(title='% delay ≥ 30 menit menurut hari & jam jadwal berangkat', xlabel='Jam (waktu lokal bandara)')
fig.colorbar(im, ax=ax, label='% delay ≥ 30 menit')
save(fig, '05_heatmap_hari_jam.png')

Pola yang umum: delay **menumpuk makin sore/malam** (efek berantai pesawat yang sama, *late aircraft*).
Jam berangkat jadi kandidat fitur kuat → di-encode siklik (sin/cos) di `03_model`.

## 9. Jarak tempuh

In [ ]:
d = con.sql('''
SELECT LEAST(Distance, 3000) // 250 * 250 AS jarak, COUNT(*) AS flights,
       100 * AVG((pm89_class >= 1)::INT) AS pct_delay30
FROM ops GROUP BY 1 ORDER BY 1
''').df()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(d.jarak, d.pct_delay30, color=BLUE, marker='o')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title='% delay ≥ 30 menit menurut jarak', xlabel='Jarak (mil, kelompok 250; ≥3000 digabung)', ylabel='% delay ≥ 30')
save(fig, '06_jarak.png')

## 10. Penyebab delay (hanya EDA — bukan fitur)

In [ ]:
c = con.sql('''
SELECT SUM(CarrierDelay) AS Maskapai, SUM(WeatherDelay) AS Cuaca, SUM(NASDelay) AS "Sistem Navigasi (NAS)",
       SUM(SecurityDelay) AS Keamanan, SUM(LateAircraftDelay) AS "Pesawat datang terlambat"
FROM ops WHERE pm89_class >= 1
''').df().T[0].sort_values()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(c.index, 100 * c / c.sum(), color=[GREY, GREY, BLUE2, BLUE2, ORANGE][:len(c)])
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title='Kontribusi menit delay menurut penyebab (penerbangan kelas ≥1)', xlabel='% total menit delay')
save(fig, '07_penyebab.png')

## 11. Delay berangkat vs delay tiba

In [ ]:
con.sql('SELECT ROUND(corr(DepDelay, ArrDelay), 3) AS korelasi_dep_arr FROM ops WHERE ArrDelay IS NOT NULL').df()

Korelasi tinggi → memakai delay **berangkat** sebagai dasar kelas PM 89 tetap konsisten dengan pengalaman penumpang saat tiba.

## 12. Simpan ringkasan

In [ ]:
total = con.execute('SELECT COUNT(*) FROM flights').fetchone()[0]
n_ops = int(cls.n.sum())
eda = {
    'rows_total': int(total),
    'rows_operated': n_ops,
    'airlines': int(con.execute('SELECT COUNT(DISTINCT Reporting_Airline) FROM ops').fetchone()[0]),
    'origins': int(con.execute('SELECT COUNT(DISTINCT Origin) FROM ops').fetchone()[0]),
    'pct_delay30': round(100 * cls.n[cls.kelas >= 1].sum() / n_ops, 2),
    'class_counts': {int(k): int(v) for k, v in zip(cls.kelas, cls.n)},
    'class_pct': {int(k): float(v) for k, v in zip(cls.kelas, cls.pct)},
    'imbalance_ratio_0_vs_5': float(cls.rasio_vs_kelas_0.iloc[-1]),
}
with open(os.path.join(BASE, 'eda_stats.json'), 'w') as f:
    json.dump(eda, f, indent=2)
eda

## 13. Angka detail untuk paper (bagian 3.1)

In [ ]:
hour = con.sql('SELECT dep_hour AS jam, ROUND(100 * AVG((pm89_class >= 1)::INT), 2) AS pct FROM ops GROUP BY 1 ORDER BY 1').df()
cell = hd.stack().sort_values(ascending=False)
detail = {
    'bulan_tertinggi': {str(k.date())[:7]: round(v, 2) for k, v in m.nlargest(6, 'pct_delay30').set_index('bulan').pct_delay30.items()},
    'bulan_terendah': {str(k.date())[:7]: round(v, 2) for k, v in m.nsmallest(6, 'pct_delay30').set_index('bulan').pct_delay30.items()},
    'pct_delay30_per_jam': dict(zip(hour.jam.astype(int).astype(str), hour.pct)),
    'hari_jam_tertinggi': [f'dow={d} jam={h}: {v:.2f}%' for (d, h), v in cell.head(5).items()],
    'pct_delay30_per_maskapai': dict(zip(a.maskapai, a.pct_delay30)),
    'pct_menit_per_penyebab': {k: round(float(v), 2) for k, v in (100 * c / c.sum()).sort_values(ascending=False).items()},
}
with open(os.path.join(BASE, 'eda_detail.json'), 'w') as f:
    json.dump(detail, f, indent=2)
detail

## Kesimpulan untuk pemodelan (diisi setelah dijalankan)

- Fitur kandidat: maskapai, bandara asal & tujuan, bulan, hari, jam (sin/cos), jarak, CRSElapsedTime.
- **Tidak dipakai** (bocoran): DepDelay, ArrDelay, *Delay penyebab, Cancelled, Diverted.
- Imbalance kelas 5 → stratified split, class weight (S4), metrik macro F1 + recall kelas 5.